# Load dependencies

In [23]:
import os
import re
from typing import List
from dotenv import load_dotenv
from collections import defaultdict
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from sentence_transformers import SentenceTransformer
from langchain_core.prompts import ChatPromptTemplate

# Load database(Ensure the corresponding database file is in the specified path.)

In [7]:
class LocalEmbedding:
    def __init__(self, model_name="BAAI/bge-large-zh-v1.5"):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return self.model.encode(texts, normalize_embeddings=True).tolist()

    def embed_query(self, text: str) -> List[float]:
        return self.model.encode([text], normalize_embeddings=True)[0].tolist()

# Run this to get the database (if you dont have)

In [ ]:
file_path = "hk01_scam_articles.md"
with open(file_path, "r", encoding="utf-8") as f:
    content = f.read()

# 按 "---" 分割文章（支持前后有空行）
articles = [a.strip() for a in re.split(r'\n-{3,}\n', content) if a.strip()]

print(f"共加载 {len(articles)} 篇诈骗新闻")
docs = []
for i, art in enumerate(articles):
    lines = art.split('\n')

    title_line = lines[0].strip() if lines else ""
    if not title_line:
        continue
    
    title_match = re.search(r'\[(.*?)\]', title_line)
    title = title_match.group(1) if title_match else title_line.replace('##', '').strip()
    
    url_match = re.search(r'https?://[^\s\)]+', title_line)
    url = url_match.group(0) if url_match else ""
    
    body_lines = lines[1:]
    body_text = '\n'.join(body_lines) 
    
    # 按 \n\n 切分
    paragraphs = [p.strip() for p in body_text.split('\n\n') if p.strip()]
    
    for j, para in enumerate(paragraphs):
        doc = Document(
            page_content=para,
            metadata={
                "source": file_path,
                "article_id": i + 1,
                "paragraph_id": j + 1,
                "title": title,
                "url": url
            }
        )
        docs.append(doc)

print(f" 共生成 {len(docs)} 个段落文档")

embedding = LocalEmbedding() # BAAI/bge-large-zh-v1.5
vectorstore = Chroma.from_documents(
    docs,
    embedding,
    persist_directory="./chroma_hk01_scam_dbtest_BAAI",
    collection_metadata={"hnsw:space": "cosine"}
)

# If you already have the database, Run this

In [26]:
embedding = LocalEmbedding() # BAAI/bge-large-zh-v1.5

vectorstore = Chroma(
    persist_directory="./chroma_hk01_scam_dbtest_BAAI",
    embedding_function=embedding,
    collection_metadata={"hnsw:space": "cosine"}
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Load LLM settings

In [14]:
load_dotenv()
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")

if not DASHSCOPE_API_KEY:
    raise ValueError("请在 .env 文件中设置 DASHSCOPE_API_KEY")

In [19]:
llm = ChatOpenAI(
    model="qwen3.5-flash",
    openai_api_key=DASHSCOPE_API_KEY,
    openai_api_base="https://dashscope.aliyuncs.com/compatible-mode/v1",
    temperature=0.0
)

In [22]:
prompt_template = """
你是一名反诈专家。请根据以下用户描述,近期真实诈骗新闻片段以及，判断用户描述的是否属于已知诈骗模式。

用户描述：
{sms}

相关近期诈骗新闻片段（来自 HK01）：
{retrieved_doc}

片段所属新闻：
{article}

请重点分析：
- 是否涉及相同手法（如“验证身份”、“紧急转账”、“中奖”等）
- 是否模仿官方机构（银行、公安、快递）
- 是否诱导点击链接/下载APP/提供验证码/拨打陌生电话
- 是否符合常见诈骗特征如：诱惑性高回报/制造紧急恐慌/非接触式联络/要求资金转账/提供第三方链接

请严格按照以下 JSON 格式输出（不要任何其他文字）：
{{"score": 整数（0-10）, "reason": "简要理由（50字内）"}}
"""
prompt = ChatPromptTemplate.from_template(prompt_template)

In [24]:
def predict_fraud_probability(sms_text: str) -> tuple[float, str]:
    retrieved_docs = retriever.invoke(sms_text)
    docs_with_scores = vectorstore.similarity_search_with_score(sms_text, k=3)
    
    # When the similarity is too low, RAG should not be used to avoid recalling irrelevant segments that could affect the judgment.
    min_distence = min(score for _, score in docs_with_scores) if docs_with_scores else 1.0
    max_sim = 1 - min_distence
    if max_sim < 0.4:
        formatted_prompt = prompt.format(sms=sms_text, retrieved_doc="无", article="无")
    else:
        para = "\n".join([doc.page_content for doc, _ in docs_with_scores])
        
        # Avoid duplication when similar segments come from the same article.
        article_groups = defaultdict(list)
        for doc, distance in docs_with_scores: 
            aid = doc.metadata["article_id"]
            cos_sim = 1.0 - distance  # 转为余弦相似度
            article_groups[aid].append((doc, cos_sim))
        
        news_parts = []
        for aid, docs_in_article in article_groups.items():
            first_doc, _ = docs_in_article[0]
            title = first_doc.metadata.get("title", "无标题")
            url = first_doc.metadata.get("url", "")
            combined_para = "\n\n".join(
                f"[相似度: {cos_sim:.2f}] {doc.page_content}"
                for doc, cos_sim in docs_in_article
            )
            news_parts.append(
                f"【新闻 #{aid}】\n标题：{title}\n链接：{url}\n\n相关段落：\n{combined_para}"
            )
        
        news = "\n---\n".join(news_parts)
        formatted_prompt = prompt.format(sms=sms_text, retrieved_doc=para, article=news)    
    
    response = llm.invoke(formatted_prompt)
    output = response.content.strip()
    
    try:
        result = json.loads(output)
        score = int(result["score"])
        reason = result.get("reason", "无理由").strip()
        score = max(0, min(10, score))
        probability = score / 10.0
        return probability, reason, news
    except Exception as e:
        numbers = re.findall(r'\d+', output)
        score = int(numbers[0]) if numbers else 5
        score = max(0, min(10, score))
        probability = score / 10.0
        
        reason_match = re.search(r'(?<=[：:"])([^"]+?)(?="|$)', output)
        if reason_match:
            reason = reason_match.group(1).strip()[:60]
        else:
            reason = f"（LLM 返回：{output[:30]}...）"
        
        return probability, reason

# Test

In [27]:
test_sms = """我是一个花店老板，有一个高中老师来找我订花，他说下个月是校庆，要购买鲜花找我预定了30盆，但是他说太忙了没时间过来看，就加了我WhatsApp我发照片给他选。
然后要交货的那天他又说叫我帮忙购买一些横幅说要搞得隆重一点。"""
prob, reason, new = predict_fraud_probability(test_sms)

print(f"短信内容：{test_sms}")
print(f"诈骗概率：{prob:.1%}")
print(f"理由：{reason}")
print(f"相关新闻报道：\n{new}")

短信内容：我是一个花店老板，有一个高中老师来找我订花，他说下个月是校庆，要购买鲜花找我预定了30盆，但是他说太忙了没时间过来看，就加了我WhatsApp我发照片给他选。
然后要交货的那天他又说叫我帮忙购买一些横幅说要搞得隆重一点。
诈骗概率：90.0%
理由：冒充校方人员批量订花，要求代买物品，非接触联络，符合新闻中冒充学校职员诈骗特征，风险极高。
相关新闻报道：
【新闻 #4】
标题：3. 騙徒扮兩間中學職員向店舖訂貨　玩具店憶始末　代購佛跳牆露端倪
链接：https://www.hk01.com/突發/60315323/騙徒扮兩間中學職員向店舖訂貨-玩具店憶始末-代購佛跳牆露端倪

相关段落：
[相似度: 0.73] 《香港01》記者向花店東主陳先生了解，他憶述一名自稱是華仁書院程主任的男子來電，訂購350支花，總數4250元，「我哋學校㗎，我畀張卡片你呀」。惟因金額過大，陳先生致電校方求證，「佢訂咁大量，我就打電話去華仁詢問處」，惟校方否認有此職員，陳先生遂回電該男子稱：「接唔到呀，做唔到你生意」。該男子追問可否訂購少一點，惟陳先生未有理會，相信對方是騙徒，「好多行家都接觸過」。
---
【新闻 #325】
标题：324. 深圳寫真館陷阱｜數百元拍BB相變買5萬元會籍　想追討店家已走佬
链接：https://www.hk01.com/大國小事/60277465/深圳寫真館陷阱-數百元拍bb相變買5萬元會籍-想追討店家已走佬

相关段落：
[相似度: 0.60] 到店後，一位被稱為藝術總監的人接待她，給她介紹店內提供的各種風格照片及價格，並表示兩百多的套餐只能拍範圍很小的風格。小林當下就對這種事前不說好的行為很反感，但也自我pua一下，覺得也很合理，畢竟套餐這麼便宜。接着對方滔滔不絕地介紹各種正價套餐：

[相似度: 0.59] 見小林不上當，店長立刻開始套近乎，一會說這原本是一萬多的價格，現在8000很划算了，一會又說下次進店可以免費送兩套服裝，還送500的指定化妝師和500的指定攝影師的資格……大概是臉皮薄的原因，小林雖然內心很不舒服，但和對方溝通時仍保持了禮貌，沒想到反而讓對方得寸進尺，挖空心思想要攻破小林的防線，一個小時過去了，對方主動價格壓到了3000多元。或許是受夠了店長、銷售的語言轟炸，加上拍攝照片確實很好看，小林最終還是鬆口付了錢。等到逃離選片室